In [1]:
!pip install pennylane matplotlib

zsh:1: command not found: pip


# Quadratic Unconstrained Binary Optimization (QUBO) with PennyLane

Quadratic Unconstrained Binary Optimization (QUBO) aims to minimize a quadratic function of binary variables $x_i \in \{0, 1\}$:
$$f(x_1, \dots, x_n) = \sum_{i} q_i x_i + \sum_{i < j} q_{ij} x_i x_j$$

To solve a QUBO on a gate-based quantum computer, we map the binary variables $x_i \in \{0, 1\}$ to spin variables $s_i \in \{1, -1\}$ using the substitution:
$$x_i = \frac{I - Z_i}{2}$$
where $Z_i$ is the Pauli-Z operator acting on qubit $i$.

## Problem 1: Simple 1-Variable QUBO
Minimize:
$$f(x_0) = -2x_0$$
Minimum is at $x_0 = 1$.

Mapping:
$$H = -I + Z_0$$

In [2]:
import pennylane as qml
from pennylane import numpy as np

dev1 = qml.device("default.qubit", wires=1)
H1 = qml.Hamiltonian([-1.0, 1.0], [qml.Identity(0), qml.PauliZ(0)])

@qml.qnode(dev1)
def circuit1(theta):
    qml.RX(theta, wires=0)
    return qml.expval(H1)

theta = np.array(0.1, requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.2)

for step in range(30):
    theta = opt.step(circuit1, theta)

print(f"Optimal theta: {theta}")
print(f"Minimum expectation value: {circuit1(theta)}")

Optimal theta: 3.043463033020032
Minimum expectation value: -1.9951891511175142


## Problem 2: Simple 2-Variable QUBO
Minimize:
$$f(x_0, x_1) = x_0 + 2x_1 - 4x_0 x_1$$
Minimum is at $x_0 = 1, x_1 = 1$.

Mapping:
$$H = 0.5I + 0.5Z_0 - Z_0 Z_1$$

In [3]:
dev2 = qml.device("default.qubit", wires=2)
H2 = qml.Hamiltonian(
    [0.5, 0.5, -1.0],
    [qml.Identity(0), qml.PauliZ(0), qml.PauliZ(0) @ qml.PauliZ(1)]
)

@qml.qnode(dev2)
def circuit2(params):
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    return qml.expval(H2)

params = np.array([0.1, 0.1], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.1)

for step in range(100):
    params = opt.step(circuit2, params)

print(f"Optimal params: {params}")
print(f"Minimum expectation value: {circuit2(params)}")

Optimal params: [2.86728758e+00 2.65873139e-06]
Minimum expectation value: -0.9813068349017255


## Problem 3: Max-Cut on a Graph
For a graph $G=(V,E)$, we want to maximize the cut size, which is equivalent to minimizing:
$$H = \sum_{(i,j) \in E} Z_i Z_j$$

We use a 3-node triangle graph: edges $(0, 1)$, $(1, 2)$, $(2, 0)$.

In [4]:
dev3 = qml.device("default.qubit", wires=3)

H3 = qml.Hamiltonian(
    [1.0, 1.0, 1.0],
    [
        qml.PauliZ(0) @ qml.PauliZ(1),
        qml.PauliZ(1) @ qml.PauliZ(2),
        qml.PauliZ(2) @ qml.PauliZ(0)
    ]
)

@qml.qnode(dev3)
def circuit3(params):
    for i in range(3):
        qml.RY(params[i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.expval(H3)

params3 = np.array([0.1, 0.1, 0.1], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.1)

for step in range(100):
    params3 = opt.step(circuit3, params3)

print(f"Optimal params: {params3}")
print(f"Minimum expectation value: {circuit3(params3)}")

Optimal params: [0.1        2.78635622 2.78635622]
Minimum expectation value: -0.996101781098404


## Problem 4: Number Partitioning
Partition a set $S = \{1, 2, 3\}$ into two subsets with equal sums. 
This is equivalent to minimizing $(1s_0 + 2s_1 + 3s_2)^2$ for $s_i \in \{1, -1\}$.

Mapping to spin variables ($s_i \rightarrow Z_i$):
$$H = 14I + 4Z_0 Z_1 + 6Z_0 Z_2 + 12Z_1 Z_2$$

In [5]:
dev4 = qml.device("default.qubit", wires=3)

H4 = qml.Hamiltonian(
    [14.0, 4.0, 6.0, 12.0],
    [
        qml.Identity(0),
        qml.PauliZ(0) @ qml.PauliZ(1),
        qml.PauliZ(0) @ qml.PauliZ(2),
        qml.PauliZ(1) @ qml.PauliZ(2)
    ]
)

@qml.qnode(dev4)
def circuit4(params):
    for i in range(3):
        qml.RY(params[i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.expval(H4)

params4 = np.array([0.1, 0.1, 0.1], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.05)

for step in range(150):
    params4 = opt.step(circuit4, params4)

print(f"Optimal params: {params4}")
print(f"Minimum expectation value: {circuit4(params4)}")

Optimal params: [1.00000000e-01 1.76885461e-07 3.14159265e+00]
Minimum expectation value: 3.197442310920451e-14


## Problem 5: Minimum Vertex Cover
For a line graph $0 - 1 - 2$, we want to find the smallest set of vertices that covers all edges.
The Hamiltonian is:
$$H = 2(1-x_0-x_1+x_0 x_1) + 2(1-x_1-x_2+x_1 x_2) + (x_0 + x_1 + x_2)$$
with $x_i = \frac{I - Z_i}{2}$.

In [6]:
dev5 = qml.device("default.qubit", wires=3)

# Calculating weights for Pauli terms:
# H = 2.5*I + 0.5*Z0 - 0.5*Z1 + 0.5*Z2 + 0.5*Z0@Z1 + 0.5*Z1@Z2
H5 = qml.Hamiltonian(
    [2.5, 0.5, -0.5, 0.5, 0.5, 0.5],
    [
        qml.Identity(0),
        qml.PauliZ(0),
        qml.PauliZ(1),
        qml.PauliZ(2),
        qml.PauliZ(0) @ qml.PauliZ(1),
        qml.PauliZ(1) @ qml.PauliZ(2)
    ]
)

@qml.qnode(dev5)
def circuit5(params):
    for i in range(3):
        qml.RY(params[i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    return qml.expval(H5)

params5 = np.array([0.1, 0.1, 0.1], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.1)

for step in range(100):
    params5 = opt.step(circuit5, params5)

print(f"Optimal params: {params5}")
print(f"Minimum expectation value: {circuit5(params5)}")

Optimal params: [0.01705093 0.01705093 3.13981721]
Minimum expectation value: 2.0001469182806484


## Problem 6: Graph 2-Coloring
Color a 4-node ring graph with 2 colors such that connected nodes have different colors.
We minimize:
$$H = Z_0 Z_1 + Z_1 Z_2 + Z_2 Z_3 + Z_3 Z_0$$

In [7]:
dev6 = qml.device("default.qubit", wires=4)

H6 = qml.Hamiltonian(
    [1.0, 1.0, 1.0, 1.0],
    [
        qml.PauliZ(0) @ qml.PauliZ(1),
        qml.PauliZ(1) @ qml.PauliZ(2),
        qml.PauliZ(2) @ qml.PauliZ(3),
        qml.PauliZ(3) @ qml.PauliZ(0)
    ]
)

@qml.qnode(dev6)
def circuit6(params):
    for i in range(4):
        qml.RY(params[i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3])
    return qml.expval(H6)

params6 = np.array([0.1, 0.1, 0.1, 0.1], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.1)

for step in range(120):
    params6 = opt.step(circuit6, params6)

print(f"Optimal params: {params6}")
print(f"Minimum expectation value: {circuit6(params6)}")

Optimal params: [0.1        3.14159265 3.14159265 3.14159265]
Minimum expectation value: -4.0
